# Auditoria de Qualidade de Respostas de um Assistente de IA de Suporte — Case Fictício

Seguiremos abaixo a seguinte estratégia lógica:

1) Leitura do brief do case (texto simulado, como se extraído de um documento de instruções), focada em identificar programaticamente uma instrução ambígua/oculta no enunciado
2) Carga e visualização inicial da base sintética de tickets de suporte respondidos por um assistente de IA, incluindo tratamento de valores nulos
3) Investigação de outliers nas variáveis numéricas e análise de correlação entre elas
4) Decisão de tratamento da instrução oculta, mantendo a base original e uma base ajustada
5) Análise descritiva do custo estimado de atendimento e do tempo de resolução, com foco em concentração por região, cliente e modelo de IA
6) Análise de cumprimento de SLA (prazo de resolução), incluindo evolução mensal e concentração de atrasos

Passo 1: Importando as bibliotecas e pacotes necessários

In [0]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Passo 2: Leitura do brief do case (simula o texto extraído de um documento de instruções)

In [0]:
# O texto abaixo simula o enunciado que um analista receberia por e-mail/PDF.
# Propositalmente, contém uma instrução com espaçamento irregular no meio do
# parágrafo — o mesmo tipo de "armadilha" que passaria despercebida numa
# leitura rápida, mas que pode mudar o resultado da análise se não for notada.
BRIEF_CASE = """
Leia com atenção:
Você recebeu um extrato de 200 interações de um assistente de IA de suporte
ao cliente. Cada linha representa um ticket respondido automaticamente pelo
assistente, com categoria, região do cliente, cliente, modelo de IA utilizado,
tempo de resolução, tamanho da resposta gerada, custo estimado do atendimento
e nota de satisfação do cliente.

Objetivo: produzir uma análise descritiva da qualidade e do custo das
respostas, identificando tickets fora do padrão (outliers), possíveis
relações entre o tamanho da resposta gerada e a satisfação do cliente, e
concentrações de custo e de descumprimento de SLA por região, cliente e
modelo de IA.

[No ta interna sobre pro ces sa men to: tickets da categoria 'Financeiro' marcados
como 'revisao_pendente' devem ser reclassificados como 'Financeiro-Prioritario'
antes da análise agregada]

Entregue o código comentado e as conclusões de forma objetiva.
"""

print(BRIEF_CASE)

Aqui é importante ressaltar que identificamos uma "instrução oculta" pois o trecho "[No ta interna sobre pro ces sa men to: tickets da categoria 'Financeiro' marcados como 'revisao_pendente' devem ser reclassificados como 'Financeiro-Prioritario' antes da análise agregada]" está com espaçamento interno irregular (palavras quebradas em pedaços curtos), um padrão que passa despercebido numa leitura rápida do brief mas ainda é identificável de forma programática.

In [0]:
def detectar_instrucao_oculta(texto: str) -> list[dict]:
    """
    Heurística para detectar palavras quebradas por espaçamento interno
    irregular (ex.: 'pro ces sa men to' em vez de 'processamento') — um
    padrão usado para "esconder" uma instrução de uma leitura superficial.

    Procura, dentro de blocos marcados entre colchetes (usados neste brief
    para notas internas), sequências de 2 ou mais palavras consecutivas de
    até 3 letras — o padrão que aparece quando uma palavra normal é
    fragmentada em pedaços curtos — e reconstrói o texto original juntando
    essas sequências.
    """
    blocos_colchetes = re.findall(r"\[(.*?)\]", texto, flags=re.DOTALL)

    achados = []
    for bloco in blocos_colchetes:
        runs = re.findall(r"(?:\b\w{1,3}\b\s+){1,}\b\w{1,3}\b", bloco)
        if not runs:
            continue

        reconstruido = bloco
        for run in runs:
            junto = run.replace(" ", "").replace("\n", "")
            reconstruido = reconstruido.replace(run, junto, 1)

        achados.append({
            "original": bloco.strip(),
            "reconstruido": " ".join(reconstruido.split()),
            "fragmentos_detectados": runs,
        })

    return achados


instrucoes_suspeitas = detectar_instrucao_oculta(BRIEF_CASE)

for i, achado in enumerate(instrucoes_suspeitas, start=1):
    print(f"[ENCONTRADO {i}] Fragmentos suspeitos: {achado['fragmentos_detectados']}")
    print(f"[ENCONTRADO {i}] Reconstituído: {achado['reconstruido']}\n")

**Descritivo das variáveis**

- `categoria`: assunto do ticket (Financeiro, Técnico, Cadastro, Cobrança)
- `status`: situação operacional do ticket (resolvido, em_andamento, revisao_pendente)
- `regiao_cliente`: região do cliente que abriu o ticket
- `cliente`: conta/cliente atendido (10 contas sintéticas)
- `modelo_ia`: versão do modelo de IA que gerou a resposta (Modelo A, B ou C)
- `tamanho_resposta_chars`: tamanho da resposta gerada, em caracteres
- `tempo_resolucao_min`: tempo total de resolução do ticket, em minutos
- `custo_estimado`: custo estimado do atendimento (R$), derivado do tamanho da resposta e do tempo de resolução
- `nota_satisfacao`: nota de satisfação do cliente (1 a 5)
- `limite_mensal_tickets`: quota mensal de tickets contratada por cliente
- `status_sla`: cumprimento do prazo de resolução (DENTRO_PRAZO / FORA_PRAZO), conforme o SLA de cada categoria
- `data_abertura`: data de abertura do ticket

**Conceitos importantes**

- **Regra de Tukey**: método de detecção de outliers baseado nos quartis, marcando como atípico todo valor fora do intervalo [Q1 - 1.5·IQR, Q3 + 1.5·IQR], onde IQR = Q3 - Q1.
- **Correlação de Pearson**: mede a força e a direção da relação linear entre duas variáveis numéricas, de -1 (negativa perfeita) a +1 (positiva perfeita).
- **SLA (Service Level Agreement)**: prazo máximo acordado para a resolução de um ticket; um ticket "FORA_PRAZO" descumpriu esse prazo.

Passo 3: Carga e visualização inicial da base sintética de tickets

In [0]:
rng = np.random.default_rng(seed=42)
N = 200

categorias = rng.choice(
    ["Financeiro", "Técnico", "Cadastro", "Cobrança"],
    size=N,
    p=[0.25, 0.35, 0.20, 0.20],
)

# Status operacional do ticket — necessário para o critério mencionado na
# instrução oculta do brief ("Financeiro" + "revisao_pendente")
status_ticket = rng.choice(
    ["resolvido", "em_andamento", "revisao_pendente"],
    size=N,
    p=[0.70, 0.20, 0.10],
)

regioes = ["Norte", "Nordeste", "Centro-Oeste", "Sudeste", "Sul"]
regiao_cliente = rng.choice(regioes, size=N, p=[0.10, 0.20, 0.15, 0.40, 0.15]).astype(object)
# Simula valores ausentes de região em parte dos tickets, como ocorreria numa base real
idx_regiao_nula = rng.choice(N, size=8, replace=False)
regiao_cliente[idx_regiao_nula] = None

clientes = [f"Conta {i:02d}" for i in range(1, 11)]
p_clientes = np.array([22, 16, 12, 10, 9, 8, 7, 6, 5, 5], dtype=float)
p_clientes /= p_clientes.sum()
cliente = rng.choice(clientes, size=N, p=p_clientes)

modelos_ia = ["Modelo A", "Modelo B", "Modelo C"]
modelo_ia = rng.choice(modelos_ia, size=N, p=[0.50, 0.30, 0.20])

# Tamanho da resposta gerada pela IA (em caracteres) — maioria concentrada
# numa faixa razoável, com uma cauda de respostas anormalmente longas
tamanho_resposta = rng.normal(loc=450, scale=120, size=N).clip(min=50)
tamanho_resposta[:6] = rng.uniform(1800, 2600, size=6)  # outliers propositais

# Tempo de resolução (minutos) — a maior parte rápida, com alguns tickets
# travados/escalonados
tempo_resolucao = rng.gamma(shape=2.0, scale=3.0, size=N)
tempo_resolucao[6:10] = rng.uniform(90, 140, size=4)  # outliers propositais

# Custo estimado do atendimento (R$), combinando tamanho da resposta e tempo
# de resolução, com um pequeno ruído aleatório
ruido_custo = rng.normal(0, 5, size=N)
custo_estimado = (0.05 * tamanho_resposta + 2.0 * tempo_resolucao + ruido_custo).clip(min=5).round(2)

# Nota de satisfação (1-5), com leve correlação negativa a respostas muito longas
ruido_nota = rng.normal(0, 0.4, size=N)
nota_satisfacao = (5 - (tamanho_resposta - 450) / 500 + ruido_nota).clip(1, 5).round(1)

# Quota mensal de tickets contratada por cliente — um cliente fica sem limite
# cadastrado de propósito, para simular uma falha de cadastro
limite_por_cliente = {c: int(v) for c, v in zip(clientes, rng.integers(15, 45, size=len(clientes)))}
limite_por_cliente["Conta 05"] = np.nan
limite_mensal_tickets = np.array([limite_por_cliente[c] for c in cliente])

# SLA (prazo de resolução) — cada categoria tem um prazo alvo diferente
sla_por_categoria = {"Financeiro": 20, "Técnico": 45, "Cadastro": 30, "Cobrança": 25}
sla_alvo = np.array([sla_por_categoria[c] for c in categorias])
status_sla = np.where(tempo_resolucao > sla_alvo, "FORA_PRAZO", "DENTRO_PRAZO")

# Data de abertura do ticket, distribuída ao longo de 6 meses
data_inicio = pd.Timestamp("2025-01-01")
dias_offset = rng.integers(0, 181, size=N)
data_abertura = data_inicio + pd.to_timedelta(dias_offset, unit="D")

df = pd.DataFrame({
    "categoria": categorias,
    "status": status_ticket,
    "regiao_cliente": regiao_cliente,
    "cliente": cliente,
    "modelo_ia": modelo_ia,
    "tamanho_resposta_chars": tamanho_resposta.round(0).astype(int),
    "tempo_resolucao_min": tempo_resolucao.round(1),
    "custo_estimado": custo_estimado,
    "nota_satisfacao": nota_satisfacao,
    "limite_mensal_tickets": limite_mensal_tickets,
    "status_sla": status_sla,
    "data_abertura": data_abertura,
})

# Embaralha as linhas só para a amostra exibida não concentrar os outliers
# propositais no topo (eles foram inseridos nas primeiras posições do array)
df = df.sample(frac=1, random_state=1).reset_index(drop=True)

df.head()

In [0]:
#Exibição das informações gerais da base de dados, incluindo o número de entradas, colunas, tipos de dados e valores nulos
df.info()

Aqui podemos observar que possuímos valores nulos na coluna `regiao_cliente`. Como o restante da análise por região depende dessa coluna, vamos tratá-los explicitamente antes de seguir, em vez de deixá-los cair silenciosamente nos agrupamentos.

In [0]:
#Substituição de valores nulos em regiao_cliente por "Não informado"
df["regiao_cliente"] = df["regiao_cliente"].fillna("Não informado")

#Exibição das informações gerais da base de dados, após ajuste de valores nulos
df.info()

Passo 4: Investigação de Outliers nas Variáveis Numéricas (Tamanho da Resposta, Tempo de Resolução e Custo Estimado)

In [0]:
#Função que calcula quartis e identifica outliers pela regra de Tukey (fora de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]) para uma coluna numérica
def estatisticas_outliers(serie: pd.Series) -> pd.Series:
    serie_valida = serie.dropna()
    q1 = serie_valida.quantile(0.25)
    q2 = serie_valida.quantile(0.50)
    q3 = serie_valida.quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    outliers = (serie_valida < limite_inferior) | (serie_valida > limite_superior)
    return pd.Series({
        "Q1": q1,
        "Mediana (Q2)": q2,
        "Q3": q3,
        "Média": serie_valida.mean(),
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "qtd_tickets": serie_valida.shape[0],
        "qtd_outliers": int(outliers.sum()),
        "% outliers": round(outliers.mean() * 100, 2),
    })


colunas_analise = {
    "tamanho_resposta_chars": "Tamanho da Resposta",
    "tempo_resolucao_min": "Tempo de Resolução",
    "custo_estimado": "Custo Estimado",
}

resumo_outliers = pd.DataFrame({nome: estatisticas_outliers(df[col]) for col, nome in colunas_analise.items()}).T

#Distribuição por quartil, média e mediana
distribuicao_quartil = resumo_outliers[["Q1", "Mediana (Q2)", "Q3", "Média"]]
distribuicao_quartil

In [0]:
#Contagem e percentual de outliers por variável (fora dos limites de Tukey), em tabela separada da distribuição por quartil
tabela_outliers = resumo_outliers[["limite_inferior", "limite_superior", "qtd_tickets", "qtd_outliers", "% outliers"]]
tabela_outliers

Observamos um cenário de outliers relevante: cerca de 3% dos tickets têm respostas anormalmente longas e cerca de 4% têm tempo de resolução muito acima do esperado — e, como o custo estimado deriva diretamente dessas duas variáveis, ele também carrega uma cauda de outliers. São candidatos prioritários de revisão de qualidade e de custo.

Passo 5: Análise de Correlação

In [0]:
correlacao = df[["tamanho_resposta_chars", "tempo_resolucao_min", "custo_estimado", "nota_satisfacao"]].corr()
correlacao.round(2)

A correlação entre tamanho da resposta e nota de satisfação é negativa e relevante (em torno de -0.87): quanto maior a resposta gerada pela IA, menor tende a ser a nota de satisfação. Já o custo estimado, por construção, acompanha de perto o tamanho da resposta e o tempo de resolução — ou seja, respostas mais longas não custam mais barato nem geram mais satisfação, apenas custam mais.

In [0]:
# Aqui vamos calcular as correlações do tempo de resolução com os demais KPIs (incluindo
# variáveis categóricas via dummies) para entender melhor qual o ponto de maior impacto
df_dummies = pd.get_dummies(df[["categoria", "modelo_ia", "regiao_cliente"]], drop_first=False)
correlacao_tempo = (
    pd.concat([df[["tamanho_resposta_chars", "custo_estimado", "nota_satisfacao"]], df_dummies], axis=1)
    .corrwith(df["tempo_resolucao_min"])
    .drop(labels=[], errors="ignore")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
correlacao_tempo.head(3).to_frame("correlacao_com_tempo_resolucao")

O top 3 de correlações com o tempo de resolução envolve majoritariamente categoria e modelo de IA (com correlações pouco significativas), o que reforça o fato de que o tempo de resolução varia bastante ticket a ticket e não segue uma regra de negócio clara atrelada a essas dimensões.

OBS: Aqui estamos deixando a critério do colaborador a aplicação da instrução oculta pois a base de dados que será utilizada nas análises seguintes será definida nos blocos abaixo.

In [0]:
#Aplicação da instrução oculta identificada no brief: reclassificação de tickets
#"Financeiro" + "revisao_pendente" para a categoria "Financeiro-Prioritario"
categoria_ajustada = df["categoria"].copy()
mask_instrucao = (df["categoria"] == "Financeiro") & (df["status"] == "revisao_pendente")
categoria_ajustada[mask_instrucao] = "Financeiro-Prioritario"

print(pd.DataFrame({"categoria": df["categoria"], "categoria_ajustada": categoria_ajustada})[mask_instrucao])

In [0]:
#Armazenando a base original em df e a base com a instrução oculta aplicada em df_ajustado
df_ajustado = df.copy()
df_ajustado["categoria"] = categoria_ajustada

print(df["categoria"].value_counts())
print()
print(df_ajustado["categoria"].value_counts())

In [0]:
#Definição de qual dataframe será utilizado nas análises seguintes: df (original) ou df_ajustado (com a instrução oculta aplicada)
df_analise = df  # troque para df_ajustado caso deseje aplicar a reclassificação de "Financeiro" + "revisao_pendente"

df_analise.head()

Passo 6: Análise Descritiva do Custo Estimado de Atendimento

In [0]:
#Big numbers: principais indicadores gerais da operação de suporte
big_numbers = pd.Series({
    "Total de Tickets": df_analise.shape[0],
    "Total de Clientes": df_analise["cliente"].nunique(),
    "Total de Regiões": df_analise["regiao_cliente"].nunique(),
    "Total de Modelos de IA": df_analise["modelo_ia"].nunique(),
    "Total de Categorias": df_analise["categoria"].nunique(),
    "Custo Estimado Total (R$)": f"{df_analise['custo_estimado'].sum():,.2f}",
    "Custo Estimado Dentro do Prazo (R$)": f"{df_analise.loc[df_analise['status_sla'] == 'DENTRO_PRAZO', 'custo_estimado'].sum():,.2f}",
    "Custo Estimado Fora do Prazo (R$)": f"{df_analise.loc[df_analise['status_sla'] == 'FORA_PRAZO', 'custo_estimado'].sum():,.2f}",
    "Nota Média de Satisfação": round(df_analise["nota_satisfacao"].mean(), 2),
}, name="Valor").to_frame()

big_numbers

Como o custo estimado de atendimento é o KPI mais próximo de uma "receita/despesa" real dessa operação, daqui em diante focaremos boa parte das análises nesse número para comunicar concentrações relevantes de custo.

Distribuição do Custo Estimado e Total de Tickets por Região, Cliente e Modelo de IA

In [0]:
#Função que resume quantidade de tickets e custo estimado total por uma dimensão (região, cliente ou modelo de IA)
def tabela_distribuicao(coluna_grupo):
    tabela = (
        df_analise.groupby(coluna_grupo)
        .agg(qtd_tickets=("categoria", "size"), custo_total_estimado=("custo_estimado", "sum"))
    )
    tabela["% tickets"] = (tabela["qtd_tickets"] / tabela["qtd_tickets"].sum() * 100).round(2)
    tabela["% custo_estimado"] = (tabela["custo_total_estimado"] / tabela["custo_total_estimado"].sum() * 100).round(2)
    return tabela.sort_values("custo_total_estimado", ascending=False)


por_regiao = tabela_distribuicao("regiao_cliente")
por_regiao

In [0]:
#Gráfico de distribuição do custo estimado por Região do Cliente
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(por_regiao.index, por_regiao["custo_total_estimado"], color="#2E5C8A")
ax.set_title("Custo Estimado Total por Região do Cliente")
ax.set_ylabel("Custo estimado total (R$)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [0]:
por_cliente = tabela_distribuicao("cliente")
por_cliente

In [0]:
#Gráfico de distribuição do custo estimado por Cliente
fig, ax = plt.subplots(figsize=(7, 5))
ordenado = por_cliente.sort_values("custo_total_estimado")
ax.barh(ordenado.index, ordenado["custo_total_estimado"], color="#2E5C8A")
ax.set_title("Custo Estimado Total por Cliente")
ax.set_xlabel("Custo estimado total (R$)")
plt.tight_layout()
plt.show()

In [0]:
por_modelo = tabela_distribuicao("modelo_ia")
por_modelo

In [0]:
#Gráfico de distribuição do custo estimado por Modelo de IA
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(por_modelo.index, por_modelo["custo_total_estimado"], color="#2E5C8A")
ax.set_title("Custo Estimado Total por Modelo de IA")
ax.set_ylabel("Custo estimado total (R$)")
plt.tight_layout()
plt.show()

Distribuição do Tempo Médio de Resolução por Região, Cliente e Modelo de IA

In [0]:
#Função que resume o tempo médio de resolução por uma dimensão (região, cliente ou modelo de IA)
def tabela_tempo_medio(coluna_grupo):
    tabela = (
        df_analise.groupby(coluna_grupo)
        .agg(qtd_tickets=("categoria", "size"), tempo_medio_resolucao=("tempo_resolucao_min", "mean"))
        .round(2)
        .sort_values("tempo_medio_resolucao", ascending=False)
    )
    return tabela


tempo_por_regiao = tabela_tempo_medio("regiao_cliente")
tempo_por_regiao

In [0]:
#Gráfico do tempo médio de resolução por Região do Cliente
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(tempo_por_regiao.index, tempo_por_regiao["tempo_medio_resolucao"], color="#2E5C8A")
ax.set_title("Tempo Médio de Resolução por Região do Cliente")
ax.set_ylabel("Tempo médio de resolução (min)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [0]:
tempo_por_cliente = tabela_tempo_medio("cliente")
tempo_por_cliente

In [0]:
#Gráfico do tempo médio de resolução por Cliente
fig, ax = plt.subplots(figsize=(7, 5))
ordenado = tempo_por_cliente.sort_values("tempo_medio_resolucao")
ax.barh(ordenado.index, ordenado["tempo_medio_resolucao"], color="#2E5C8A")
ax.set_title("Tempo Médio de Resolução por Cliente")
ax.set_xlabel("Tempo médio de resolução (min)")
plt.tight_layout()
plt.show()

In [0]:
tempo_por_modelo = tabela_tempo_medio("modelo_ia")
tempo_por_modelo

In [0]:
#Gráfico do tempo médio de resolução por Modelo de IA
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(tempo_por_modelo.index, tempo_por_modelo["tempo_medio_resolucao"], color="#2E5C8A")
ax.set_title("Tempo Médio de Resolução por Modelo de IA")
ax.set_ylabel("Tempo médio de resolução (min)")
plt.tight_layout()
plt.show()

Aqui temos uma observação um pouco contraintuitiva: o modelo de IA com maior custo total (Passo 6) nem sempre é o de maior tempo médio de resolução — o custo é puxado tanto pelo tempo quanto pelo tamanho da resposta, então um modelo pode gerar respostas mais longas em menos tempo e ainda assim custar mais caro.

Quota Mensal de Tickets e Percentual de Utilização por Cliente

In [0]:
#Quota mensal contratada por cliente e percentual de utilização (tickets realizados / quota contratada)
#Clientes com quota não cadastrada aparecem com "% utilizacao_limite" nulo
limite_por_cliente_tab = df_analise.drop_duplicates("cliente").set_index("cliente")["limite_mensal_tickets"]
qtd_por_cliente = df_analise.groupby("cliente").size()

utilizacao_cliente = pd.DataFrame({
    "qtd_tickets_realizados": qtd_por_cliente,
    "limite_mensal_contratado": limite_por_cliente_tab,
})
utilizacao_cliente["% utilizacao_limite"] = (
    utilizacao_cliente["qtd_tickets_realizados"] / utilizacao_cliente["limite_mensal_contratado"] * 100
).round(1)

utilizacao_cliente.sort_values("% utilizacao_limite", ascending=False)

In [0]:
#Gráfico do percentual de utilização da quota mensal por Cliente, considerando todos os clientes
fig, ax = plt.subplots(figsize=(7, 5))
ordenado = utilizacao_cliente.sort_values("% utilizacao_limite")
ax.barh(ordenado.index, ordenado["% utilizacao_limite"].fillna(0), color="#2E5C8A")
ax.axvline(100, color="#C0392B", linestyle="--", linewidth=1)
ax.set_title("Percentual de Utilização da Quota Mensal por Cliente")
ax.set_xlabel("% da quota mensal utilizada")
plt.tight_layout()
plt.show()

Aqui vale um ponto significativo de alerta: a "Conta 05" não possui quota mensal cadastrada, porém realiza tickets normalmente — uma falha de cadastro que merece correção antes de qualquer decisão comercial baseada nesse indicador. Além disso, alguns clientes ultrapassam 100% da quota contratada, sinalizando a necessidade de uma regra de controle em tempo real (e não apenas de acompanhamento retroativo, como feito aqui).

Insights sobre a Distribuição de Custo Estimado e Total de Tickets

In [0]:
#Função que interpreta uma tabela de distribuição (qtd_tickets, custo_total_estimado, % tickets, % custo_estimado) e devolve insights em texto
def gerar_insights(tabela, nome_dimensao, top_n=3):
    por_custo = tabela.sort_values("custo_total_estimado", ascending=False)
    por_qtd = tabela.sort_values("qtd_tickets", ascending=False)
    ticket_medio = tabela["custo_total_estimado"] / tabela["qtd_tickets"]

    lider_custo = por_custo.index[0]
    lider_qtd = por_qtd.index[0]
    pct_lider_custo = por_custo["% custo_estimado"].iloc[0]
    pct_top_n = por_custo["% custo_estimado"].iloc[:top_n].sum()
    nivel_concentracao = "alta" if pct_top_n >= 50 else "moderada"

    print(f"Insights - {nome_dimensao}:")
    print(f"- {lider_custo} concentra a maior fatia do custo estimado da operação ({pct_lider_custo:.1f}%).")
    print(f"- Os {top_n} maiores em custo estimado somam {pct_top_n:.1f}% do total, indicando concentração {nivel_concentracao}.")
    if lider_custo == lider_qtd:
        print(f"- {lider_custo} também é quem tem mais tickets, ou seja, lidera tanto em custo quanto em quantidade.")
    else:
        print(f"- {lider_qtd} tem o maior número de tickets, mas quem mais contribui em custo estimado é {lider_custo} — sinal de tickets médios diferentes.")
    print(f"- Maior ticket médio de custo: {ticket_medio.idxmax()} (R$ {ticket_medio.max():,.2f}). Menor ticket médio de custo: {ticket_medio.idxmin()} (R$ {ticket_medio.min():,.2f}).")
    print()

gerar_insights(por_regiao, "Região do Cliente")
gerar_insights(por_cliente, "Cliente")
gerar_insights(por_modelo, "Modelo de IA")

Através da distribuição da base podemos ressaltar alguns pontos de atenção como:

1) Concentração regional e por cliente do custo estimado, o que sugere priorizar a revisão de qualidade de respostas nesses grupos específicos ao invés de uma revisão genérica em toda a base;
2) Diferenças entre os modelos de IA no ticket médio de custo, o que pode orientar a escolha de qual modelo usar em cada categoria de ticket.

Outro ponto fundamental para analisar o recorte atual da operação seria o cumprimento de SLA, nos levando a entender como a inadimplência de prazo se distribui entre região, cliente e modelo de IA.

Distribuição de Custo Estimado por Status de SLA

In [0]:
#Distribuição de custo estimado e de total de tickets por status de SLA (DENTRO_PRAZO vs. FORA_PRAZO)
cores_sla = {"DENTRO_PRAZO": "#2E5C8A", "FORA_PRAZO": "#C0392B"}

dist_sla = (
    df_analise.groupby("status_sla")
    .agg(qtd_tickets=("categoria", "size"), custo_total_estimado=("custo_estimado", "sum"))
)
dist_sla["% tickets"] = (dist_sla["qtd_tickets"] / dist_sla["qtd_tickets"].sum() * 100).round(2)
dist_sla["% custo_estimado"] = (dist_sla["custo_total_estimado"] / dist_sla["custo_total_estimado"].sum() * 100).round(2)
dist_sla

In [0]:
#Gráfico de distribuição do custo estimado por status de SLA
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.bar(dist_sla.index, dist_sla["custo_total_estimado"], color=[cores_sla[s] for s in dist_sla.index])
ax.set_title("Custo Estimado Total por Status de SLA")
ax.set_ylabel("Custo estimado total (R$)")
plt.tight_layout()
plt.show()

Evolução Mensal do Custo Estimado por Status de SLA

In [0]:
#Evolução mensal (considerando a data de abertura): linha com o custo total estimado e barras com a contagem de tickets
evolucao_mensal = (
    df_analise.assign(mes=df_analise["data_abertura"].dt.to_period("M").astype(str))
    .groupby("mes")
    .agg(custo_total_estimado=("custo_estimado", "sum"), qtd_tickets=("categoria", "size"))
    .sort_index()
)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()
ax2.bar(evolucao_mensal.index, evolucao_mensal["qtd_tickets"], color="#AAB7B8", alpha=0.6, label="Qtd. tickets")
ax1.plot(evolucao_mensal.index, evolucao_mensal["custo_total_estimado"], color="#2E5C8A", marker="o", label="Custo estimado (R$)")
ax1.set_ylabel("Custo estimado total (R$)")
ax2.set_ylabel("Quantidade de tickets")
ax1.set_title("Evolução Mensal do Custo Estimado e Volume de Tickets")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Evolução Mensal do Custo Estimado — Recorte dos Meses Mais Significativos

In [0]:
#Recorte dos meses mais significativos: mantém apenas meses cuja quantidade de tickets representa ao menos 1% do total da base
limite_relevancia = df_analise.shape[0] * 0.01
evolucao_relevante = evolucao_mensal[evolucao_mensal["qtd_tickets"] >= limite_relevancia]
evolucao_relevante

Distribuição de Descumprimento de SLA por Região, Cliente e Modelo de IA

In [0]:
#Função que resume a quantidade e o custo em tickets fora do prazo de SLA por uma dimensão
def tabela_atraso(coluna_grupo):
    base = df_analise.assign(em_atraso=df_analise["status_sla"] == "FORA_PRAZO")
    tabela = base.groupby(coluna_grupo).agg(
        qtd_tickets=("categoria", "size"),
        qtd_atraso=("em_atraso", "sum"),
        custo_total_estimado=("custo_estimado", "sum"),
    )
    custo_atraso = base[base["em_atraso"]].groupby(coluna_grupo)["custo_estimado"].sum()
    tabela["custo_atraso"] = custo_atraso.reindex(tabela.index).fillna(0)
    tabela["% tickets_atraso"] = (tabela["qtd_atraso"] / tabela["qtd_tickets"] * 100).round(1)
    tabela["% custo_atraso"] = (tabela["custo_atraso"] / tabela["custo_total_estimado"] * 100).round(1)
    return tabela.sort_values("% custo_atraso", ascending=False)


atraso_regiao = tabela_atraso("regiao_cliente")
atraso_regiao

In [0]:
#Gráfico de distribuição do custo em atraso vs. dentro do prazo por Região do Cliente
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(atraso_regiao.index, atraso_regiao["% custo_atraso"], color="#C0392B")
ax.set_title("% do Custo Estimado Fora do Prazo por Região do Cliente")
ax.set_ylabel("% do custo estimado fora do prazo")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [0]:
atraso_cliente = tabela_atraso("cliente")
atraso_cliente

In [0]:
#Gráfico de distribuição do custo em atraso vs. dentro do prazo por Cliente
fig, ax = plt.subplots(figsize=(7, 5))
ordenado = atraso_cliente.sort_values("% custo_atraso")
ax.barh(ordenado.index, ordenado["% custo_atraso"], color="#C0392B")
ax.set_title("% do Custo Estimado Fora do Prazo por Cliente")
ax.set_xlabel("% do custo estimado fora do prazo")
plt.tight_layout()
plt.show()

In [0]:
atraso_modelo = tabela_atraso("modelo_ia")
atraso_modelo

In [0]:
#Gráfico de distribuição do custo em atraso vs. dentro do prazo por Modelo de IA
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(atraso_modelo.index, atraso_modelo["% custo_atraso"], color="#C0392B")
ax.set_title("% do Custo Estimado Fora do Prazo por Modelo de IA")
ax.set_ylabel("% do custo estimado fora do prazo")
plt.tight_layout()
plt.show()

Insights sobre a Distribuição de Descumprimento de SLA

In [0]:
#Taxa geral de descumprimento de SLA da operação, usada como referência de comparação nos insights por grupo
taxa_geral_tickets_atraso = (df_analise["status_sla"] == "FORA_PRAZO").mean() * 100
taxa_geral_custo_atraso = (
    df_analise.loc[df_analise["status_sla"] == "FORA_PRAZO", "custo_estimado"].sum()
    / df_analise["custo_estimado"].sum() * 100
)

#Função que interpreta uma tabela de atraso de SLA e devolve insights em texto. Grupos com poucos tickets
#(< qtd_tickets_min) são ignorados na liderança do ranking para não destacar taxas estatisticamente frágeis
def gerar_insights_sla(tabela, nome_dimensao, qtd_tickets_min=10):
    relevante = tabela[tabela["qtd_tickets"] >= qtd_tickets_min]
    por_custo = relevante.sort_values("% custo_atraso", ascending=False)
    por_tickets = relevante.sort_values("% tickets_atraso", ascending=False)

    pior_custo = por_custo.index[0]
    melhor_custo = por_custo.index[-1]
    pior_tickets = por_tickets.index[0]

    print(f"Insights de SLA - {nome_dimensao} (considerando apenas grupos com {qtd_tickets_min}+ tickets):")
    print(f"- {pior_custo} tem a maior taxa de descumprimento de SLA em custo: {por_custo['% custo_atraso'].iloc[0]:.1f}% do custo estimado está fora do prazo, bem acima da média da operação ({taxa_geral_custo_atraso:.1f}%).")
    print(f"- {melhor_custo} tem a menor taxa de descumprimento de SLA em custo: {por_custo['% custo_atraso'].iloc[-1]:.1f}%.")
    if pior_tickets == pior_custo:
        print(f"- {pior_tickets} também lidera em quantidade de tickets fora do prazo ({por_tickets['% tickets_atraso'].iloc[0]:.1f}%), reforçando o risco concentrado nesse grupo.")
    else:
        print(f"- Em quantidade de tickets, quem mais descumpre o prazo é {pior_tickets} ({por_tickets['% tickets_atraso'].iloc[0]:.1f}%).")
    print()

print(f"Taxa geral de descumprimento de SLA da operação: {taxa_geral_tickets_atraso:.1f}% dos tickets e {taxa_geral_custo_atraso:.1f}% do custo estimado.")
print()
gerar_insights_sla(atraso_regiao, "Região do Cliente")
gerar_insights_sla(atraso_cliente, "Cliente")
gerar_insights_sla(atraso_modelo, "Modelo de IA")

Conclusão

Em suma, podemos observar que existem concentrações relevantes de custo e de descumprimento de SLA na base, o que deve ser encarado como sinal de alerta, com foco no desenvolvimento de um plano de ação cobrindo as diversas frentes — região do cliente, cliente e modelo de IA — visando uma operação de suporte mais previsível e eficiente em custo. Em contrapartida, essa mesma análise auxilia no desenho de critérios de priorização de revisão de qualidade e na escolha de qual modelo de IA usar em cada categoria de ticket.

Outros pontos de atenção válidos para uma análise mais detalhada são a ausência de quota mensal cadastrada para um dos clientes (nesse caso, a Conta 05) e a falta de um SLA único e padronizado entre categorias. No primeiro caso, vale automatizar, via sistema, a regra de cadastro de quota para evitar falhas como essa. Já quanto ao SLA por categoria, o observado (percentual relevante de tickets fora do prazo em alguns grupos) sugere que o ideal seria revisar os prazos-alvo por categoria e/ou associá-los a região e modelo de IA, reduzindo o desequilíbrio nessa frente.

Próximos Passos

1) Governança de SLA e Quota: desenho de uma regra de negócio clara para o SLA por categoria, associada a região e/ou modelo de IA; validação sistêmica no cadastro de quota mensal por cliente, evitando inconsistências como a identificada na Conta 05; e controle de consumo da quota em tempo real, para evitar extrapolações significativas.

2) Dashboard automatizado: ingestão recorrente do extrato de tickets via um pipeline de dados, alimentando esta mesma análise em um dashboard interativo atualizado em tempo real, em vez de um notebook rodado manualmente.

3) Chat de consulta (RAG): um chat interno, alimentado pela base de tickets e por documentos de política de atendimento, permitindo que áreas não técnicas perguntem em linguagem natural — por exemplo, "quais clientes têm maior risco de descumprimento de SLA hoje?".

4) Modelo de risco (Machine Learning): um modelo preditivo treinado nos padrões de descumprimento de SLA já identificados (região, modelo de IA, categoria, cliente) para estimar o risco de novos tickets antes mesmo de existir histórico de resolução, alimentando uma interface simples de consulta no momento da abertura do ticket.

Observação!!

Aqui toda a análise foi feita sem utilizar a aplicação da "instrução oculta" (reclassificar "Financeiro" + "revisao_pendente" para "Financeiro-Prioritario"). Se ajustarmos `df_analise` para `df_ajustado` no Passo 5, as análises voltadas para categoria precisariam passar por revisão de insights e conclusão.